# 05 · Final pipeline
Starts from the best model of 01–04 and separates two kinds of changes:
* **A. Inference / data strategy, same weights** – test resolution, flip/scale TTA, tiled (sliced) inference,
  temporal score smoothing (bonus).
* **B. Model change** – retrain with a stronger configuration (bigger backbone and / or more frames).

Every step is measured with the same evaluator; the final submission uses the best combination.

Kaggle: Accelerator **GPU**, **Internet on**. Add the output of notebook 03 as input (it contains `weights/`).

## 0. Setup

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/YOUR_USERNAME/YOUR_REPO.git'  # <- change once: your public repo with src/common.py
FAST_DEV = os.environ.get('FAST_DEV') == '1'                # tiny smoke test on a fake dataset; keep off on Kaggle

if not any(Path(d, 'common.py').exists() for d in ['src', '../src', 'repo/src']):
    subprocess.run(['git', 'clone', '-q', REPO_URL, 'repo'], check=True)
SRC = next(d for d in ['src', '../src', 'repo/src'] if Path(d, 'common.py').exists())
sys.path.insert(0, SRC)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.208', 'pycocotools'], check=True)

import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
from common import *

seed_everything()
ROOT = find_data_root()
TMP = Path('/tmp/hw2') if not FAST_DEV else Path('work_dev')
OUT = Path('results'); OUT.mkdir(exist_ok=True)
FIG = Path('images'); FIG.mkdir(exist_ok=True)
WEIGHTS = Path('weights'); WEIGHTS.mkdir(exist_ok=True)
STEP = 5

## 1. Best model so far
Set `BASE_NAME` / `BASE_IMGSZ` to the best run from `results/metrics.csv`.
The weights are searched in `weights/` and in attached Kaggle inputs (output of notebook 03).

In [ ]:
BASE_NAME, BASE_IMGSZ = 'yolo_p2_1024', 1024


def find_weights(name):
    for pattern in [f'weights/{name}.pt', f'/kaggle/input/*/weights/{name}.pt', f'/kaggle/input/*/*/weights/{name}.pt']:
        hits = sorted(Path('/').glob(pattern.lstrip('/'))) if pattern.startswith('/') else sorted(Path('.').glob(pattern))
        if hits:
            return hits[0]
    raise FileNotFoundError(f'{name}.pt not found: attach notebook 03 output or copy it to weights/')


base_model = YOLO(find_weights(BASE_NAME))
val_frames = list_frames(ROOT / 'val', STEP)
evaluator = CocoEvaluator(ROOT, val_frames)
steps = []  # (step name, metrics) for the incremental table


def record(name, preds, fps, part):
    m = evaluator.evaluate(preds)
    steps.append(dict(step=name, part=part, fps=fps, **{k: m[k] for k in ['mAP', 'AP50', 'AP_small', 'AP_medium', 'AR500']}))
    print(steps[-1])
    return m

base_preds, fps = predict_ultralytics(base_model, val_frames, imgsz=BASE_IMGSZ)
record(f'{BASE_NAME} @ {BASE_IMGSZ}', base_preds, fps, 'baseline')

## 2A. Inference strategies on the same weights

In [ ]:
# test-time resolution (model trained at BASE_IMGSZ; larger input = small objects get more pixels)
for sz in [1280, 1536] if not FAST_DEV else [384]:
    p, fps = predict_ultralytics(base_model, val_frames, imgsz=sz)
    record(f'imgsz {sz}', p, fps, 'A: inference')

In [ ]:
# flip + multi-scale TTA (Ultralytics augment=True)
tta_preds, fps = predict_ultralytics(base_model, val_frames, imgsz=BASE_IMGSZ, augment=True)
record(f'TTA @ {BASE_IMGSZ}', tta_preds, fps, 'A: inference')

In [ ]:
# tiled inference: 2x2 overlapping tiles + full frame, merged by class-wise NMS (SAHI idea)
tiled_preds, fps = predict_ultralytics_tiled(base_model, val_frames, imgsz=BASE_IMGSZ, grid=(2, 2), overlap=0.2)
record(f'tiles 2x2 + full @ {BASE_IMGSZ}', tiled_preds, fps, 'A: inference')

### Temporal score smoothing (bonus: data temporality)
Needs consecutive frames, so it is evaluated on **all** frames of the 7 val clips (not every 5th).
alpha = 1 means no smoothing; the best alpha / IoU threshold is chosen on val.

In [ ]:
val_all = list_frames(ROOT / 'val', 1)
ev_all = CocoEvaluator(ROOT, val_all)
p_all, _ = predict_ultralytics(base_model, val_all, imgsz=BASE_IMGSZ)
rows = []
for alpha in [1.0, 0.8, 0.6, 0.4]:
    for iou_thr in [0.3, 0.5]:
        m = ev_all.evaluate(temporal_rescore(p_all, val_all, alpha=alpha, iou_thr=iou_thr) if alpha < 1 else p_all)
        rows.append(dict(alpha=alpha, iou_thr=iou_thr, **{k: m[k] for k in ['mAP', 'AP50', 'AP_small']}))
        if alpha == 1.0:
            break
temporal = pd.DataFrame(rows)
temporal.to_csv(OUT / 'temporal_rescoring.csv', index=False)
print(temporal.to_string(index=False))
BEST_T = temporal.sort_values('mAP').iloc[-1]
print('best temporal setting:', dict(BEST_T))

## 2B. Model change (retraining) — run if GPU time allows
* `yolo_m_p2_1024` – YOLOv8m + P2 head (bigger backbone), same data and epochs as notebook 03.
Compare its row with the baseline row to separate the model effect from the inference effects above.

In [ ]:
TRAIN_B = False  # set True to run (~2-3 h on a T4)
if TRAIN_B or FAST_DEV:
    yaml_path = prepare_training_copy(ROOT, TMP / 'data', step_train=STEP, step_val=STEP)
    name = 'yolo_m_p2_1024'
    model = YOLO('yolov8m-p2.yaml' if not FAST_DEV else 'yolov8n-p2.yaml')
    model.load('yolov8m.pt' if not FAST_DEV else 'yolov8n.pt')
    model.train(data=str(yaml_path), name=name, epochs=15 if not FAST_DEV else 1, imgsz=1024 if not FAST_DEV else 320,
                batch=4, seed=SEED, deterministic=True, cos_lr=True, close_mosaic=3, workers=4,
                project=str((TMP / 'runs').resolve()), exist_ok=True, verbose=False, device=0 if torch.cuda.is_available() else 'cpu')
    run_dir = Path(model.trainer.save_dir)
    shutil.copy(run_dir / 'weights' / 'best.pt', WEIGHTS / f'{name}.pt')
    shutil.copy(run_dir / 'results.csv', OUT / f'ultra_{name}.csv')
    m_model = YOLO(WEIGHTS / f'{name}.pt')
    p, fps = predict_ultralytics(m_model, val_frames, imgsz=BASE_IMGSZ)
    m = record(f'{name} (model change)', p, fps, 'B: model')
    log_result(dict(model=name, family='one-stage anchor-free', epochs=15, step=STEP, imgsz=1024, fps=fps, **m), OUT / 'metrics.csv')

## 3. Incremental table

In [ ]:
steps_df = pd.DataFrame(steps)
steps_df.to_csv(OUT / 'final_pipeline_steps.csv', index=False)
print(steps_df.to_string(index=False))

## 4. Summary of all detectors (from results/metrics.csv of every notebook)
Copy each notebook's `results/` into the repo; this cell merges whatever is there.

In [ ]:
frames = []
for d in {OUT, Path(SRC).parent / 'results'}:
    if (d / 'metrics.csv').exists():
        frames.append(pd.read_csv(d / 'metrics.csv'))
if frames:
    allm = pd.concat(frames).drop_duplicates(subset=['model'], keep='last')
    cols = ['model', 'family', 'mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AR500', 'fps']
    print(allm[cols].sort_values('mAP', ascending=False).to_string(index=False))
    allm.to_csv(OUT / 'summary_all_models.csv', index=False)

## 5. Final submission
Choose the inference setting from section 3 (and the temporal setting from 2A if it helped).

In [ ]:
FINAL_WEIGHTS = BASE_NAME          # or 'yolo_m_p2_1024' if section 2B won
FINAL_MODE = 'tiles'               # 'plain' | 'tta' | 'tiles'  <- pick the best row of section 3
USE_TEMPORAL = bool(BEST_T.alpha < 1)

final_model = YOLO(find_weights(FINAL_WEIGHTS) if FINAL_WEIGHTS == BASE_NAME else WEIGHTS / f'{FINAL_WEIGHTS}.pt')
tf = test_frames(ROOT)
if FINAL_MODE == 'tiles':
    test_preds, _ = predict_ultralytics_tiled(final_model, tf, imgsz=BASE_IMGSZ, grid=(2, 2), overlap=0.2)
else:
    test_preds, _ = predict_ultralytics(final_model, tf, imgsz=BASE_IMGSZ, augment=FINAL_MODE == 'tta')
if USE_TEMPORAL:
    test_preds = temporal_rescore(test_preds, tf, alpha=BEST_T.alpha, iou_thr=BEST_T.iou_thr)
write_submission(test_preds, ROOT, 'submission_final.csv')